In [5]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import KFold, train_test_split, cross_val_predict, cross_val_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pyswarm import pso
from scipy.stats import wilcoxon


In [6]:
HOME = os.path.expanduser("~")
SAVE_DIR = os.path.join(HOME, "Desktop", "Trabalho3")
os.makedirs(SAVE_DIR, exist_ok=True)
print("Arquivos serão salvos em:", SAVE_DIR)

METRICS_CSV = os.path.join(SAVE_DIR, "metrics_all.csv")
PARAMS_CSV  = os.path.join(SAVE_DIR, "best_params_all.csv")
FINAL_CSV   = os.path.join(SAVE_DIR, "final_results.csv")


Arquivos serão salvos em: /home/nathalya/Desktop/Trabalho3


In [7]:
#carregar dados
df_mat = pd.read_csv('student-mat.csv', sep=';')
df_por = pd.read_csv('student-por.csv', sep=';')
df = pd.concat([df_mat, df_por], ignore_index=True)

In [8]:
#pré-processamento
cat_cols = [c for c in df.columns if df[c].dtype == 'object']
for c in cat_cols:
    df[c] = LabelEncoder().fit_transform(df[c].astype(str))

X_all = df.drop(columns=['G3'])
y_all = df['G3']

selector = SelectKBest(f_regression, k=10)
selector.fit(X_all, y_all)
selected_features = list(X_all.columns[selector.get_support()])
X_select = X_all[selected_features]
scaler_for_caseB = StandardScaler()
X_select_scaled = pd.DataFrame(scaler_for_caseB.fit_transform(X_select), columns=X_select.columns)

print("Selected features (k=10):", selected_features)

Selected features (k=10): ['school', 'age', 'Medu', 'Fedu', 'studytime', 'failures', 'higher', 'Dalc', 'G1', 'G2']


In [9]:
#Funções de avaliação 
def svm_loss_for_pyswarm(x, X, y, kf):
    """
    x: vetor 1D [C, gamma]
    retorna: escalar -mean(R2) (a pso minimiza)
    """
    C = float(x[0])
    gamma = float(x[1])
    C = max(1e-8, C)
    gamma = max(1e-12, gamma)

    model = SVR(C=C, gamma=gamma, kernel='rbf')
    scores = cross_val_score(model, X, y, cv=kf, scoring='r2', n_jobs=None)
    return -scores.mean()


def rfr_loss_for_pyswarm(x, X, y, kf):
    """
    x: vetor 1D [n_estimators, max_depth, min_samples_split]
    retorna: escalar -mean(R2)
    """
    n_estimators = int(round(x[0])); n_estimators = max(10, n_estimators)
    max_depth = int(round(x[1])); md = None if max_depth <= 0 else int(max_depth)
    min_samples_split = int(round(x[2])); min_samples_split = max(2, min_samples_split)

    model = RandomForestRegressor(n_estimators=n_estimators,
                                  max_depth=md,
                                  min_samples_split=min_samples_split,
                                  random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X, y, cv=kf, scoring='r2', n_jobs=None)
    return -scores.mean()


In [ ]:
 #BOUNDS 

svm_lb = [0.1, 1e-4]
svm_ub = [1000.0, 1.0]

# RFR
rfr_lb = [10, -1, 2]
rfr_ub = [500, 100, 50]

#parâmetros pso 
swarmsize = 20   
maxiter = 30     


In [ ]:
#CSVs 
if os.path.exists(METRICS_CSV):
    df_metrics = pd.read_csv(METRICS_CSV)
    start_run = len(df_metrics)
    print(f"Arquivo {METRICS_CSV} encontrado — continuando a partir do run {start_run}.")
else:
    cols = [
        'run',
        'svm_orig_R2','svm_orig_MAE','svm_orig_MSE',
        'svm_prep_R2','svm_prep_MAE','svm_prep_MSE',
        'rfr_orig_R2','rfr_orig_MAE','rfr_orig_MSE',
        'rfr_prep_R2','rfr_prep_MAE','rfr_prep_MSE'
    ]
    df_metrics = pd.DataFrame(columns=cols)
    start_run = 0
    print("Nenhum CSV encontrado. Começando do run 0.")

if os.path.exists(PARAMS_CSV):
    df_params = pd.read_csv(PARAMS_CSV)
else:
    cols_p = [
        'run',
        'svm_orig_C','svm_orig_gamma',
        'svm_prep_C','svm_prep_gamma',
        'rfr_orig_n_estimators','rfr_orig_max_depth','rfr_orig_min_samples_split',
        'rfr_prep_n_estimators','rfr_prep_max_depth','rfr_prep_min_samples_split'
    ]
    df_params = pd.DataFrame(columns=cols_p)



Arquivo /home/nathalya/Desktop/Trabalho3/metrics_all.csv encontrado — continuando a partir do run 30.


In [ ]:
N_EXEC = 30
block_size = 5
if start_run >= N_EXEC:
    print("Já existem 30+ runs concluídos")
else:
    end_run = min(start_run + block_size, N_EXEC)
    print(f"Executando runs de {start_run} até {end_run-1} (total alvo {N_EXEC})")

    for run in range(start_run, end_run):
        print(f"\n--- RUN {run+1}/{N_EXEC} ---")
        kf = KFold(n_splits=10, shuffle=True, random_state=run)

        # SVM original 
        np.random.seed(run)  
        try:
            best_pos_svm, best_val_svm = pso(
                svm_loss_for_pyswarm,
                svm_lb, svm_ub,
                args=(X_all.values, y_all.values, kf),
                swarmsize=swarmsize, maxiter=maxiter, debug=False
            )
            C_best, gamma_best = float(best_pos_svm[0]), float(best_pos_svm[1])
        except Exception as e:
            print("pso (SVM orig) falhou:", e)
            C_best, gamma_best = 1.0, 1e-3

        model_svm_orig = SVR(C=C_best, gamma=gamma_best, kernel='rbf')
        y_pred_svm_orig = cross_val_predict(model_svm_orig, X_all.values, y_all.values, cv=kf)
        svm_orig_R2 = r2_score(y_all.values, y_pred_svm_orig)
        svm_orig_MAE = mean_absolute_error(y_all.values, y_pred_svm_orig)
        svm_orig_MSE = mean_squared_error(y_all.values, y_pred_svm_orig)
        print(f"SVM orig: R2={svm_orig_R2:.4f} (C={C_best:.3f}, gamma={gamma_best:.6f})")

        #SVM pre-processado 
        np.random.seed(run + 1000)
        try:
            best_pos_svm2, best_val_svm2 = pso(
                svm_loss_for_pyswarm,
                svm_lb, svm_ub,
                args=(X_select_scaled.values, y_all.values, kf),
                swarmsize=swarmsize, maxiter=maxiter, debug=False
            )
            C_b, gamma_b = float(best_pos_svm2[0]), float(best_pos_svm2[1])
        except Exception as e:
            print("pso (SVM prep) falhou:", e)
            C_b, gamma_b = 1.0, 1e-3

        model_svm_prep = SVR(C=C_b, gamma=gamma_b, kernel='rbf')
        y_pred_svm_prep = cross_val_predict(model_svm_prep, X_select_scaled.values, y_all.values, cv=kf)
        svm_prep_R2 = r2_score(y_all.values, y_pred_svm_prep)
        svm_prep_MAE = mean_absolute_error(y_all.values, y_pred_svm_prep)
        svm_prep_MSE = mean_squared_error(y_all.values, y_pred_svm_prep)
        print(f"SVM prep: R2={svm_prep_R2:.4f} (C={C_b:.3f}, gamma={gamma_b:.6f})")

        #RFR original
        np.random.seed(run + 2000)
        try:
            best_pos_rfr, best_val_rfr = pso(
                rfr_loss_for_pyswarm,
                rfr_lb, rfr_ub,
                args=(X_all.values, y_all.values, kf),
                swarmsize=swarmsize, maxiter=maxiter, debug=False
            )
            n_est = int(round(best_pos_rfr[0])); max_d = int(round(best_pos_rfr[1])); min_s = int(round(best_pos_rfr[2]))
        except Exception as e:
            print("pso (RFR orig) falhou:", e)
            n_est, max_d, min_s = 100, 10, 2

        n_est = max(10, n_est)
        min_s = max(2, min_s)
        md_val = None if max_d <= 0 else max_d
        model_rfr_orig = RandomForestRegressor(n_estimators=n_est, max_depth=md_val, min_samples_split=min_s, random_state=42, n_jobs=-1)
        y_pred_rfr_orig = cross_val_predict(model_rfr_orig, X_all.values, y_all.values, cv=kf)
        rfr_orig_R2 = r2_score(y_all.values, y_pred_rfr_orig)
        rfr_orig_MAE = mean_absolute_error(y_all.values, y_pred_rfr_orig)
        rfr_orig_MSE = mean_squared_error(y_all.values, y_pred_rfr_orig)
        print(f"RFR orig: R2={rfr_orig_R2:.4f} (n={n_est}, max_depth={md_val}, min_split={min_s})")

        #RFR preprocessado
        np.random.seed(run + 3000)
        try:
            best_pos_rfr2, best_val_rfr2 = pso(
                rfr_loss_for_pyswarm,
                rfr_lb, rfr_ub,
                args=(X_select_scaled.values, y_all.values, kf),
                swarmsize=swarmsize, maxiter=maxiter, debug=False
            )
            n_est2 = int(round(best_pos_rfr2[0])); max_d2 = int(round(best_pos_rfr2[1])); min_s2 = int(round(best_pos_rfr2[2]))
        except Exception as e:
            print("pso (RFR prep) falhou:", e)
            n_est2, max_d2, min_s2 = 100, 10, 2

        n_est2 = max(10, n_est2)
        min_s2 = max(2, min_s2)
        md_val2 = None if max_d2 <= 0 else max_d2
        model_rfr_prep = RandomForestRegressor(n_estimators=n_est2, max_depth=md_val2, min_samples_split=min_s2, random_state=42, n_jobs=-1)
        y_pred_rfr_prep = cross_val_predict(model_rfr_prep, X_select_scaled.values, y_all.values, cv=kf)
        rfr_prep_R2 = r2_score(y_all.values, y_pred_rfr_prep)
        rfr_prep_MAE = mean_absolute_error(y_all.values, y_pred_rfr_prep)
        rfr_prep_MSE = mean_squared_error(y_all.values, y_pred_rfr_prep)
        print(f"RFR prep: R2={rfr_prep_R2:.4f} (n={n_est2}, max_depth={md_val2}, min_split={min_s2})")

        #apped metrics 
        row = {
            'run': run,
            'svm_orig_R2': svm_orig_R2, 'svm_orig_MAE': svm_orig_MAE, 'svm_orig_MSE': svm_orig_MSE,
            'svm_prep_R2': svm_prep_R2, 'svm_prep_MAE': svm_prep_MAE, 'svm_prep_MSE': svm_prep_MSE,
            'rfr_orig_R2': rfr_orig_R2, 'rfr_orig_MAE': rfr_orig_MAE, 'rfr_orig_MSE': rfr_orig_MSE,
            'rfr_prep_R2': rfr_prep_R2, 'rfr_prep_MAE': rfr_prep_MAE, 'rfr_prep_MSE': rfr_prep_MSE
        }

        df_metrics = pd.concat([df_metrics, pd.DataFrame([row])], ignore_index=True)

        #append params 
        rowp = {
            'run': run,
            'svm_orig_C': C_best, 'svm_orig_gamma': gamma_best,
            'svm_prep_C': C_b, 'svm_prep_gamma': gamma_b,
            'rfr_orig_n_estimators': n_est, 'rfr_orig_max_depth': md_val, 'rfr_orig_min_samples_split': min_s,
            'rfr_prep_n_estimators': n_est2, 'rfr_prep_max_depth': md_val2, 'rfr_prep_min_samples_split': min_s2
        }

        df_params = pd.concat([df_params, pd.DataFrame([rowp])], ignore_index=True)

       
        df_metrics.to_csv(METRICS_CSV, index=False)
        df_params.to_csv(PARAMS_CSV, index=False)
        print(f"Run {run+1} salvo em CSV.")

In [ ]:
if os.path.exists(METRICS_CSV):
    df_metrics = pd.read_csv(METRICS_CSV)

if len(df_metrics) >= 30:
    print("\n30 runs completos — executando holdout final e testes.")

    #Holdout
    X_train_all, X_test_all, y_train, y_test = train_test_split(X_all, y_all, test_size=0.20, random_state=42)
    X_train_prep, X_test_prep, _, _ = train_test_split(X_select_scaled, y_all, test_size=0.20, random_state=42)

    #escolher melhor (R2)
    idx_best_svm_orig = int(df_metrics['svm_orig_R2'].astype(float).idxmax())
    idx_best_svm_prep = int(df_metrics['svm_prep_R2'].astype(float).idxmax())
    idx_best_rfr_orig = int(df_metrics['rfr_orig_R2'].astype(float).idxmax())
    idx_best_rfr_prep = int(df_metrics['rfr_prep_R2'].astype(float).idxmax())

    #obter params 
    df_params = pd.read_csv(PARAMS_CSV)
    best_svm_orig = df_params.loc[df_params['run']==idx_best_svm_orig].iloc[0]
    best_svm_prep = df_params.loc[df_params['run']==idx_best_svm_prep].iloc[0]
    best_rfr_orig = df_params.loc[df_params['run']==idx_best_rfr_orig].iloc[0]
    best_rfr_prep = df_params.loc[df_params['run']==idx_best_rfr_prep].iloc[0]

    #treinar e avaliar holdout
    model_svm_final_orig = SVR(C=float(best_svm_orig['svm_orig_C']), gamma=float(best_svm_orig['svm_orig_gamma']))
    model_svm_final_orig.fit(X_train_all.values, y_train.values)
    yhat = model_svm_final_orig.predict(X_test_all.values)
    svm_orig_hold_R2 = r2_score(y_test.values, yhat); svm_orig_hold_MAE = mean_absolute_error(y_test.values, yhat)

    model_svm_final_prep = SVR(C=float(best_svm_prep['svm_prep_C']), gamma=float(best_svm_prep['svm_prep_gamma']))
    model_svm_final_prep.fit(X_train_prep.values, y_train.values)
    yhat = model_svm_final_prep.predict(X_test_prep.values)
    svm_prep_hold_R2 = r2_score(y_test.values, yhat); svm_prep_hold_MAE = mean_absolute_error(y_test.values, yhat)

    model_rfr_final_orig = RandomForestRegressor(n_estimators=int(best_rfr_orig['rfr_orig_n_estimators']),
                                                max_depth=(None if pd.isna(best_rfr_orig['rfr_orig_max_depth']) else int(best_rfr_orig['rfr_orig_max_depth'])),
                                                min_samples_split=int(best_rfr_orig['rfr_orig_min_samples_split']),
                                                random_state=42, n_jobs=-1)
    model_rfr_final_orig.fit(X_train_all.values, y_train.values)
    yhat = model_rfr_final_orig.predict(X_test_all.values)
    rfr_orig_hold_R2 = r2_score(y_test.values, yhat); rfr_orig_hold_MAE = mean_absolute_error(y_test.values, yhat)

    model_rfr_final_prep = RandomForestRegressor(n_estimators=int(best_rfr_prep['rfr_prep_n_estimators']),
                                                max_depth=(None if pd.isna(best_rfr_prep['rfr_prep_max_depth']) else int(best_rfr_prep['rfr_prep_max_depth'])),
                                                min_samples_split=int(best_rfr_prep['rfr_prep_min_samples_split']),
                                                random_state=42, n_jobs=-1)
    model_rfr_final_prep.fit(X_train_prep.values, y_train.values)
    yhat = model_rfr_final_prep.predict(X_test_prep.values)
    rfr_prep_hold_R2 = r2_score(y_test.values, yhat); rfr_prep_hold_MAE = mean_absolute_error(y_test.values, yhat)
    import pandas as pd
    from scipy.stats import wilcoxon

  
    df = pd.read_csv(METRICS_CSV)

    svm_orig_r2 = df['svm_orig_R2'].dropna().values
    svm_prep_r2 = df['svm_prep_R2'].dropna().values
    
    rfr_orig_r2 = df['rfr_orig_R2'].dropna().values
    rfr_prep_r2 = df['rfr_prep_R2'].dropna().values
    
  
    print("\nPrimeiros 5 valores (SVM orig):", svm_orig_r2[:5])
    print("Primeiros 5 valores (SVM prep):", svm_prep_r2[:5])
    
    print("\nPrimeiros 5 valores (RFR orig):", rfr_orig_r2[:5])
    print("Primeiros 5 valores (RFR prep):", rfr_prep_r2[:5])
    
    #Wilcoxon
    stat_svm, p_svm = wilcoxon(svm_orig_r2, svm_prep_r2)
    stat_rfr, p_rfr = wilcoxon(rfr_orig_r2, rfr_prep_r2)
    
    print("\n=== Teste de Wilcoxon ===")
    print(f"SVM  - estatística: {stat_svm:.4f} | p-valor: {p_svm:.10f}")
    print(f"RFR  - estatística: {stat_rfr:.4f} | p-valor: {p_rfr:.10f}")
        #Wilcoxon (R2)
    svm_orig_R2_list = df_metrics['svm_orig_R2'].astype(float).values[:30]
    svm_prep_R2_list = df_metrics['svm_prep_R2'].astype(float).values[:30]
    rfr_orig_R2_list = df_metrics['rfr_orig_R2'].astype(float).values[:30]
    rfr_prep_R2_list = df_metrics['rfr_prep_R2'].astype(float).values[:30]

    stat_svm, p_svm = wilcoxon(svm_orig_R2_list, svm_prep_R2_list)
    stat_rfr, p_rfr = wilcoxon(rfr_orig_R2_list, rfr_prep_R2_list)

    final_results = {
        'svm_orig_hold_R2': svm_orig_hold_R2, 'svm_orig_hold_MAE': svm_orig_hold_MAE,
        'svm_prep_hold_R2': svm_prep_hold_R2, 'svm_prep_hold_MAE': svm_prep_hold_MAE,
        'rfr_orig_hold_R2': rfr_orig_hold_R2, 'rfr_orig_hold_MAE': rfr_orig_hold_MAE,
        'rfr_prep_hold_R2': rfr_prep_hold_R2, 'rfr_prep_hold_MAE': rfr_prep_hold_MAE,
        'wilcoxon_svm_stat': stat_svm, 'wilcoxon_svm_p': p_svm,
        'wilcoxon_rfr_stat': stat_rfr, 'wilcoxon_rfr_p': p_rfr,
        'best_run_idx_svm_orig': idx_best_svm_orig, 'best_run_idx_svm_prep': idx_best_svm_prep,
        'best_run_idx_rfr_orig': idx_best_rfr_orig, 'best_run_idx_rfr_prep': idx_best_rfr_prep
    }

    
    df_final = pd.DataFrame([final_results])
    df_final.to_csv(FINAL_CSV, index=False)
    print("Holdout e Wilcoxon finalizados. Resultados salvos em:", FINAL_CSV)

    print("\nResumo holdout (R2):")
    print("SVM orig:", svm_orig_hold_R2, "| SVM prep:", svm_prep_hold_R2)
    print("RFR orig:", rfr_orig_hold_R2, "| RFR prep:", rfr_prep_hold_R2)
 

else:
    print("\nAinda não completou 30 runs. Continue executando o script até completar 30 runs.")
    print("Para continuar, execute novamente o mesmo script — ele detectará o CSV e continuará automaticamente.")



30 runs completos — executando holdout final e testes.

Primeiros 5 valores (SVM orig): [0.83123783 0.83585872 0.83099266 0.83070872 0.83070882]
Primeiros 5 valores (SVM prep): [0.82734217 0.82720633 0.82717129 0.82769484 0.82729339]

Primeiros 5 valores (RFR orig): [0.85437588 0.85186785 0.85596859 0.85772566 0.86084093]
Primeiros 5 valores (RFR prep): [0.8250786  0.82577519 0.82200208 0.82569282 0.82551137]

=== Teste de Wilcoxon ===
SVM  - estatística: 0.0000 | p-valor: 0.0000000019
RFR  - estatística: 0.0000 | p-valor: 0.0000000019
Holdout e Wilcoxon finalizados. Resultados salvos em: /home/nathalya/Desktop/Trabalho3/final_results.csv

Resumo holdout (R2):
SVM orig: 0.8054076335728526 | SVM prep: 0.8022333882412841
RFR orig: 0.8227226387826192 | RFR prep: 0.780142619452389


In [21]:
# média e desvio padrão das métricas 
df = pd.read_csv("metrics_all.csv")   


medias = df.mean(numeric_only=True)
desvios = df.std(numeric_only=True)

tabela_geral = pd.DataFrame({
    "media": medias,
    "desvio_padrao": desvios
})

def estatisticas(prefixo):
    bloco = df.filter(regex=f"^{prefixo}")
    return pd.DataFrame({
        "media": bloco.mean(),
        "desvio_padrao": bloco.std()
    })

svm_orig_stats = estatisticas("svm_orig_")
svm_prep_stats = estatisticas("svm_prep_")
rfr_orig_stats = estatisticas("rfr_orig_")
rfr_prep_stats = estatisticas("rfr_prep_")


tabela_geral.to_csv("estatisticas_geral.csv", index=True)
svm_orig_stats.to_csv("estatisticas_svm_original.csv", index=True)
svm_prep_stats.to_csv("estatisticas_svm_prep.csv", index=True)
rfr_orig_stats.to_csv("estatisticas_rfr_original.csv", index=True)
rfr_prep_stats.to_csv("estatisticas_rfr_prep.csv", index=True)


print("\n===== ESTATÍSTICAS GERAIS =====")
print(tabela_geral)

print("\n===== SVM ORIGINAL =====")
print(svm_orig_stats)

print("\n===== SVM PREPROCESSADO =====")
print(svm_prep_stats)

print("\n===== RFR ORIGINAL =====")
print(rfr_orig_stats)

print("\n===== RFR PREPROCESSADO =====")
print(rfr_prep_stats)




===== ESTATÍSTICAS GERAIS =====
                  media  desvio_padrao
run           14.500000       8.803408
svm_orig_R2    0.832877       0.003271
svm_orig_MAE   0.889933       0.031421
svm_orig_MSE   2.493865       0.048813
svm_prep_R2    0.827291       0.000291
svm_prep_MAE   0.863927       0.003631
svm_prep_MSE   2.577218       0.004336
rfr_orig_R2    0.856562       0.002427
rfr_orig_MAE   0.896678       0.007471
rfr_orig_MSE   2.140425       0.036223
rfr_prep_R2    0.825369       0.002465
rfr_prep_MAE   0.971343       0.013596
rfr_prep_MSE   2.605896       0.036784

===== SVM ORIGINAL =====
                 media  desvio_padrao
svm_orig_R2   0.832877       0.003271
svm_orig_MAE  0.889933       0.031421
svm_orig_MSE  2.493865       0.048813

===== SVM PREPROCESSADO =====
                 media  desvio_padrao
svm_prep_R2   0.827291       0.000291
svm_prep_MAE  0.863927       0.003631
svm_prep_MSE  2.577218       0.004336

===== RFR ORIGINAL =====
                 media  desvio_pad